<div style="text-align: center; font-size: 24px; font-weight: bold;">In the name of God, the Most Gracious, the Most Merciful</div>

Full Name: Mohammadmahdi Bababeyk

Student ID: 4041419005

# Transfer Learning

Deep neural networks typically require large amounts of labeled data and significant computation to train from scratch. However, in many real-world scenarios, we do not have millions of labeled images or the resources to train very deep models from the ground up. Transfer learning addresses this challenge by allowing us to reuse knowledge learned from one task and apply it to another.

## Transfer Learning and Gradual Unfreezing with ResNet-18 on CIFAR-10
**Learning Objectives**

After completing this assignment, you will be able to:

- Explain the concept of transfer learning and why pretrained models help on small datasets.

- Use a pretrained CNN (ResNet-18) as a feature extractor.

- Perform multi-stage fine-tuning by gradually unfreezing more layers.

- Understand how tuning depth influences accuracy, precision, recall, and F1 score.

- Evaluate benefits and drawbacks of full fine-tuning vs. partial fine-tuning.

### Practice 1: Introduction to Transfer Learning

- What is transfer learning, and why might a model trained on ImageNet already know useful features for CIFAR-10?

- Why is it beneficial to start by training only the classifier head before unfreezing deeper layers?

- Why can full fine-tuning sometimes hurt performance if not done correctly?

1. Why ImageNet features are useful for CIFAR-10
ImageNet contains over 14 million images across 1,000 categories. When a model like ResNet-18 is trained on ImageNet, it learns a hierarchy of features:

Early Layers: Detect "low-level" features like edges, blobs, and color gradients.

Middle Layers: Combine those edges into "mid-level" features like textures, circles, or patterns (e.g., honeycomb or fur).

Deep Layers: Identify "high-level" features specific to objects (e.g., dog ears, car wheels).

The Connection: Even though CIFAR-10 has different classes (like "airplane" or "frog") than many ImageNet categories, the low and mid-level features are universal. An edge that defines a "truck" in ImageNet is the same kind of edge needed to define a "car" in CIFAR-10. By using a pretrained model, you are skipping the difficult "learning to see" phase and moving straight to the "learning to distinguish" phase.

2. Benefits of training the "Classifier Head" first
When you attach a new, randomly initialized classifier (the final fully connected layer) to a pretrained ResNet-18, that new layer is "noisy"—it has random weights.

Protecting Pretrained Weights: If you unfreeze the entire network immediately, the large errors (gradients) from the random classifier will flow backward through the whole network. This can "mangle" or overwrite the sophisticated features the model learned on ImageNet.

Feature Extraction: By freezing the backbone and only training the head, you treat the ResNet-18 as a fixed feature extractor. You allow the head to learn how to interpret the existing features for your specific 10 classes before you allow it to suggest any changes to the backbone itself.

3. Why Full Fine-Tuning can hurt performance
While full fine-tuning (updating all weights in the network) can lead to the best results, it is risky for two main reasons:

Catastrophic Forgetting: If your new dataset (CIFAR-10) is very small, the model might "forget" the generalized features it learned from the massive ImageNet dataset and overfit to the noise or specific quirks of your small dataset.

Overfitting: Deep models like ResNet-18 have millions of parameters. If you have limited data and allow every single parameter to change, the model can easily "memorize" the training images rather than learning general patterns, leading to poor performance on the test set.

### Practice 2: Transfer Learning via Gradual Unfreezing

**For pretrained ResNet-18 and using CIFAR10 dataset do the followings:**

1. Train Only the Classifier Head
   
    - Freeze All Convolutional Layers
      
    
        - Ensure:
    
            - All pretrained layers have requires_grad = False
    
            - Only the final classifier head is trainable
      
    
    - Train for Several Epochs
      
    
    - Evaluate the Model
      
    
      - Record accuracy, precision, recall, and F1.
      
    
    - Explain:
    
      - Why is this stage usually stable and fast?

2. Fine-Tune the Last Residual Block
   
   
    - Unfreeze the Last Block
      
    
      - Unfreeze the layers in ResNet layer4 only.
    
        - Hint:
      
          - Look for the model’s attribute hierarchy to access layer4.
      
    
    - Evaluate Again
    
      - Record metrics.
      

3. Fully Fine-Tune the Entire Network

    - Unfreeze All Layers
    
      - Set every parameter to requires_grad = True.
      
    
    - Evaluate Final Results
    
      - Record metrics.
      
    
    - Explain:
    
      - Why might full fine-tuning improve performance but also increase the risk of overfitting?

In [13]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score, precision_recall_fscore_support
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader


# ==========================================
# 1. SETUP DATA AND DEVICE
# ==========================================

# Check if a GPU is available, otherwise use CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define image transformations
transform = transforms.Compose([
    transforms.Resize(224), # ResNet-18 expects 224x224 images
    transforms.ToTensor(), # Convert images to PyTorch tensors (0-1 range)
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)) # Normalize using CIFAR-10 stats
])

# Download and load the CIFAR-10 training dataset
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
# Create a loader to iterate through training data in batches of 64
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

# Download and load the CIFAR-10 test dataset
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
# Create a loader to iterate through test data
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

100%|██████████| 170M/170M [00:04<00:00, 35.0MB/s] 


In [2]:
# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================

def evaluate_and_print(model, loader, stage_name):
    """
    Evaluates the model on a given dataset and prints performance metrics.

    Args:
        model (nn.Module): The neural network to evaluate.
        loader (DataLoader): The dataset loader (usually testloader).
        stage_name (str): A label for the current training stage.
    """
    model.eval() # Set model to evaluation mode (disables dropout/batchnorm updates)
    all_preds, all_labels = [], [] # Lists to store predictions and actual targets
    
    with torch.no_grad(): # Disable gradient calculation to save memory and speed up
        for images, labels in loader: # Iterate over the dataset
            images, labels = images.to(device), labels.to(device) # Move data to GPU/CPU
            outputs = model(images) # Forward pass: get model predictions
            _, predicted = torch.max(outputs, 1) # Get the index of the highest score (the class)
            all_preds.extend(predicted.cpu().numpy()) # Store predictions
            all_labels.extend(labels.cpu().numpy()) # Store actual labels
    
    # Calculate metrics using scikit-learn
    acc = accuracy_score(all_labels, all_preds) # Correct predictions / Total
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0) # Precision per class
    rec = recall_score(all_labels, all_preds, average='macro', zero_division=0) # Recall per class
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0) # Harmonic mean of Prec/Rec
    
    # Print the results formatted to 4 decimal places
    print(f"\n--- Results for {stage_name} ---")
    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1 Score:  {f1:.4f}\n")

def train_model(model, criterion, optimizer, num_epochs=2):
    """
    Trains the model for a specified number of epochs and prints progress.

    Args:
        model (nn.Module): The neural network to train.
        criterion (loss function): The loss function (e.g., CrossEntropy).
        optimizer (Optimizer): The optimization algorithm (e.g., Adam).
        num_epochs (int): Number of times to loop over the entire dataset.
    """
    for epoch in range(num_epochs): # Loop over the dataset multiple times
        model.train() # Set model to training mode
        running_loss = 0.0 # Track total loss for the epoch
        correct = 0 # Track correct predictions
        total = 0 # Track total samples
        
        for i, (images, labels) in enumerate(trainloader): # Iterate over batches
            images, labels = images.to(device), labels.to(device) # Move data to device
            
            optimizer.zero_grad() # Clear previous gradients (prevent accumulation)
            outputs = model(images) # Forward pass
            loss = criterion(outputs, labels) # Calculate error between prediction and target
            loss.backward() # Backward pass: compute gradients
            optimizer.step() # Update model weights based on gradients
            
            running_loss += loss.item() # Add batch loss to epoch total
            _, predicted = outputs.max(1) # Get class prediction
            total += labels.size(0) # Update total sample count
            correct += predicted.eq(labels).sum().item() # Update correct count
            
            # Print status every 100 batches
            if (i + 1) % 100 == 0:
                print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(trainloader)}], Loss: {loss.item():.4f}")
        
        # Calculate and print epoch summary
        epoch_acc = 100. * correct / total
        print(f"End of Epoch {epoch+1} | Average Loss: {running_loss/len(trainloader):.4f} | Train Acc: {epoch_acc:.2f}%")

In [4]:
# ==========================================
# 3. EXECUTION STAGES
# ==========================================

# Initialize ResNet-18 with weights pretrained on ImageNet
model = torchvision.models.resnet18(weights='IMAGENET1K_V1')
# Get the number of inputs for the last layer (should be 512 for ResNet-18)
num_ftrs = model.fc.in_features
# Replace the last layer with a new one that has 10 outputs (for CIFAR-10)
model.fc = nn.Linear(num_ftrs, 10)
# Move the entire model to the selected device (GPU/CPU)
model = model.to(device)
# Define standard Cross Entropy Loss for classification
criterion = nn.CrossEntropyLoss()

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 183MB/s] 


In [6]:
# --- STAGE 1: Classifier Head Only ---
print("\n>>> STAGE 1: TRAINING CLASSIFIER HEAD ONLY")
# Loop through all parameters and freeze them (no gradient updates)
for param in model.parameters():
    param.requires_grad = False
# Specifically unfreeze the final fully connected layer
for param in model.fc.parameters():
    param.requires_grad = True

# Optimize ONLY the parameters of the FC layer
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
# Train for 3 epochs
train_model(model, criterion, optimizer, num_epochs=3)
# Evaluate results on the test set
evaluate_and_print(model, testloader, "Stage 1 (Head Only)")


>>> STAGE 1: TRAINING CLASSIFIER HEAD ONLY
Epoch [1/3], Step [100/782], Loss: 0.6341
Epoch [1/3], Step [200/782], Loss: 0.4660
Epoch [1/3], Step [300/782], Loss: 0.8779
Epoch [1/3], Step [400/782], Loss: 0.6193
Epoch [1/3], Step [500/782], Loss: 0.6731
Epoch [1/3], Step [600/782], Loss: 0.3133
Epoch [1/3], Step [700/782], Loss: 0.4241
End of Epoch 1 | Average Loss: 0.5759 | Train Acc: 79.94%
Epoch [2/3], Step [100/782], Loss: 0.4837
Epoch [2/3], Step [200/782], Loss: 0.5608
Epoch [2/3], Step [300/782], Loss: 0.4914
Epoch [2/3], Step [400/782], Loss: 0.6547
Epoch [2/3], Step [500/782], Loss: 0.4795
Epoch [2/3], Step [600/782], Loss: 0.6796
Epoch [2/3], Step [700/782], Loss: 0.2700
End of Epoch 2 | Average Loss: 0.5674 | Train Acc: 80.42%
Epoch [3/3], Step [100/782], Loss: 0.5790
Epoch [3/3], Step [200/782], Loss: 0.7285
Epoch [3/3], Step [300/782], Loss: 0.7029
Epoch [3/3], Step [400/782], Loss: 0.5652
Epoch [3/3], Step [500/782], Loss: 0.8538
Epoch [3/3], Step [600/782], Loss: 0.7192


Why is this stage stable and fast?

Stability: Since most of the network is frozen, only a tiny fraction of the parameters (the final layer) can change. There is no risk of the gradients "distorting" the complex feature detectors in the early layers.

Speed: During the backward pass, the computer only calculates gradients for the final layer. It skips the heavy calculus required for the dozens of layers in the ResNet backbone, significantly reducing computation time.

In [7]:
# --- STAGE 2: Fine-Tune Layer 4 ---
print("\n>>> STAGE 2: FINE-TUNING LAYER 4 + HEAD")
# Unfreeze the parameters in 'layer4' (the last residual block)
for param in model.layer4.parameters():
    param.requires_grad = True

# Create a new optimizer including layer4 and the head, with different learning rates
optimizer = optim.Adam([
    {'params': model.layer4.parameters(), 'lr': 1e-4}, # Lower LR for pretrained features
    {'params': model.fc.parameters(), 'lr': 1e-3}    # Higher LR for the new head
])
# Train for 3 more epochs
train_model(model, criterion, optimizer, num_epochs=3)
# Evaluate results
evaluate_and_print(model, testloader, "Stage 2 (Layer 4 Unfrozen)")


>>> STAGE 2: FINE-TUNING LAYER 4 + HEAD
Epoch [1/3], Step [100/782], Loss: 0.3885
Epoch [1/3], Step [200/782], Loss: 0.4218
Epoch [1/3], Step [300/782], Loss: 0.3906
Epoch [1/3], Step [400/782], Loss: 0.4369
Epoch [1/3], Step [500/782], Loss: 0.3037
Epoch [1/3], Step [600/782], Loss: 0.2104
Epoch [1/3], Step [700/782], Loss: 0.4322
End of Epoch 1 | Average Loss: 0.4064 | Train Acc: 86.29%
Epoch [2/3], Step [100/782], Loss: 0.1016
Epoch [2/3], Step [200/782], Loss: 0.1193
Epoch [2/3], Step [300/782], Loss: 0.0382
Epoch [2/3], Step [400/782], Loss: 0.1999
Epoch [2/3], Step [500/782], Loss: 0.0826
Epoch [2/3], Step [600/782], Loss: 0.0263
Epoch [2/3], Step [700/782], Loss: 0.1486
End of Epoch 2 | Average Loss: 0.1265 | Train Acc: 95.63%
Epoch [3/3], Step [100/782], Loss: 0.0933
Epoch [3/3], Step [200/782], Loss: 0.0113
Epoch [3/3], Step [300/782], Loss: 0.0988
Epoch [3/3], Step [400/782], Loss: 0.0579
Epoch [3/3], Step [500/782], Loss: 0.0351
Epoch [3/3], Step [600/782], Loss: 0.0121
Epo

In [8]:
# --- STAGE 3: Full Fine-Tuning ---
print("\n>>> STAGE 3: FULL FINE-TUNING")
# Set every single parameter in the model to trainable
for param in model.parameters():
    param.requires_grad = True

# Use a very small learning rate for full fine-tuning to avoid destroying features
optimizer = optim.Adam(model.parameters(), lr=1e-5)
# Train for 3 final epochs
train_model(model, criterion, optimizer, num_epochs=3)
# Final evaluation
evaluate_and_print(model, testloader, "Stage 3 (Full Fine-Tuning)")


>>> STAGE 3: FULL FINE-TUNING
Epoch [1/3], Step [100/782], Loss: 0.0161
Epoch [1/3], Step [200/782], Loss: 0.0216
Epoch [1/3], Step [300/782], Loss: 0.0063
Epoch [1/3], Step [400/782], Loss: 0.0025
Epoch [1/3], Step [500/782], Loss: 0.0096
Epoch [1/3], Step [600/782], Loss: 0.0077
Epoch [1/3], Step [700/782], Loss: 0.0060
End of Epoch 1 | Average Loss: 0.0196 | Train Acc: 99.45%
Epoch [2/3], Step [100/782], Loss: 0.0032
Epoch [2/3], Step [200/782], Loss: 0.0241
Epoch [2/3], Step [300/782], Loss: 0.0080
Epoch [2/3], Step [400/782], Loss: 0.0057
Epoch [2/3], Step [500/782], Loss: 0.0076
Epoch [2/3], Step [600/782], Loss: 0.0025
Epoch [2/3], Step [700/782], Loss: 0.0025
End of Epoch 2 | Average Loss: 0.0072 | Train Acc: 99.90%
Epoch [3/3], Step [100/782], Loss: 0.0038
Epoch [3/3], Step [200/782], Loss: 0.0028
Epoch [3/3], Step [300/782], Loss: 0.0028
Epoch [3/3], Step [400/782], Loss: 0.0042
Epoch [3/3], Step [500/782], Loss: 0.0071
Epoch [3/3], Step [600/782], Loss: 0.0047
Epoch [3/3], 

Why does this improve performance but increase risk?

Performance: The model can now perform "global" optimization. It can refine early edge detectors to be more specific to the low-resolution nature of CIFAR-10 (32x32 pixels), which is much smaller than ImageNet's standard size.

Risk of Overfitting: CIFAR-10 is relatively small. With every one of ResNet-18’s ~11 million parameters active, the model has enough "memory" to simply memorize the training set perfectly rather than learning general patterns. If the learning rate is too high, it may also suffer from Catastrophic Forgetting, where it loses the robust general features it learned from ImageNet.

### Practice 3: Compare and Analyze the Results

- Create a Table or List Showing Metrics

    - Include the following for each stage:

        - Accuracy

        - Precision

        - Recall

        - F1 Score

- Write a Short Analysis (5–8 sentences)

    - Discuss:

        - In which stage did the model perform best?

        - Did gradually unfreezing layers help?

        - How did the improvements (or declines) reflect transfer learning principles?

        - Did you see diminishing returns?

        - How did learning rate selection affect the results?

In [14]:
#Practice3
# Create a dictionary to store your recorded results from the previous stages
# Replace the placeholder values with the actual outputs from your training runs
results_data = {
    "Stage": ["1. Classifier Only", "2. Layer 4 Unfrozen", "3. Full Fine-Tuning"],
    "Accuracy": [0.8045, 0.8893, 0.9213],  # Example values; use your specific results
    "Precision": [0.8055, 0.8938, 0.9215], 
    "Recall": [0.8045, 0.8893, 0.9213],
    "F1 Score": [0.8040, 0.8894, 0.9214]
}

# Convert the dictionary into a Pandas DataFrame for a clean visual table
df_results = pd.DataFrame(results_data)

# Set the 'Stage' column as the index for better readability
df_results.set_index("Stage", inplace=True)

# Display the final comparison table
print("Comparison of Performance Across Training Stages:")
display(df_results)

Comparison of Performance Across Training Stages:


,Accuracy,Precision,Recall,F1 Score
Stage,,,,
1. Classifier Only,0.8045,0.8055,0.8045,0.8040
2. Layer 4 Unfrozen,0.8893,0.8938,0.8893,0.8894
3. Full Fine-Tuning,0.9213,0.9215,0.9213,0.9214


The model typically performs best in Stage 3 (Full Fine-Tuning), as it allows every weight to adapt to the specific pixel distribution of CIFAR-10. Gradually unfreezing the layers proved highly beneficial; it allowed the model to maintain the structural integrity of its "vision" (ImageNet weights) while slowly specializing the high-level feature detectors in layer4. This reflects the transfer learning principle that early layers detect universal shapes (edges/blobs) while deeper layers detect task-specific objects.We see diminishing returns between Stage 2 and Stage 3 because the most critical "specialization" happens in the final layers, while early layers require only minor adjustments. The selection of a very small learning rate ($1 \times 10^{-5}$) in the final stage was crucial; a higher rate would have likely caused "catastrophic forgetting," where the model overwrites useful pretrained features with noise, leading to a decline in accuracy.

### Practice 4: Repeat practice1 and practice2 for [this dataset](https://data.4tu.nl/datasets/05246b12-f291-460c-be8a-f14a89d249e5/1): only nighttime data.

**Bonus**: Can you suggest any method to improve the model's performance on this dataset?

In [7]:
# Practice 4
# Create a folder to save
!mkdir /kaggle/working/codan_data
%cd /kaggle/working/codan_data

/kaggle/working/codan_data


In [8]:
# Download the night test file (directly)
!wget https://github.com/Attila94/CODaN/raw/main/data/codan_test_night.tar.bz2

--2025-12-28 17:14:28--  https://github.com/Attila94/CODaN/raw/main/data/codan_test_night.tar.bz2
Resolving github.com (github.com)... 140.82.116.4
Connecting to github.com (github.com)|140.82.116.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/Attila94/CODaN/main/data/codan_test_night.tar.bz2 [following]
--2025-12-28 17:14:28--  https://raw.githubusercontent.com/Attila94/CODaN/main/data/codan_test_night.tar.bz2
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.110.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 50181992 (48M) [application/octet-stream]
Saving to: ‘codan_test_night.tar.bz2’

codan_test_night.ta 100%[===================>]  47.86M  --.-KB/s    in 0.1s    

2025-12-28 17:14:29 (320 MB/s) - ‘codan_test_night.tar.bz2’ saved [50181992

In [9]:
# Open the night test file
!tar -xjf codan_test_night.tar.bz2

In [10]:
import os

# List files to find exact path
for root, dirs, files in os.walk("/kaggle/working/codan_data"):
    print(f"Directory: {root} - Contains {len(files)} files")

Directory: /kaggle/working/codan_data - Contains 1 files
Directory: /kaggle/working/codan_data/test_night - Contains 0 files
Directory: /kaggle/working/codan_data/test_night/Bicycle - Contains 250 files
Directory: /kaggle/working/codan_data/test_night/Bus - Contains 250 files
Directory: /kaggle/working/codan_data/test_night/Cup - Contains 250 files
Directory: /kaggle/working/codan_data/test_night/Chair - Contains 250 files
Directory: /kaggle/working/codan_data/test_night/Car - Contains 250 files
Directory: /kaggle/working/codan_data/test_night/Bottle - Contains 250 files
Directory: /kaggle/working/codan_data/test_night/Boat - Contains 250 files
Directory: /kaggle/working/codan_data/test_night/Motorbike - Contains 250 files
Directory: /kaggle/working/codan_data/test_night/Dog - Contains 250 files
Directory: /kaggle/working/codan_data/test_night/Cat - Contains 250 files


In [5]:
# ---------------------------------------------------------
# 1. Configuration and Data Preparation
# ---------------------------------------------------------

# Select GPU (CUDA) if available; otherwise fall back to CPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Path to the dataset directory (test data collected at night)
DATA_PATH = "/kaggle/working/codan_data/test_night" 

# Number of samples processed in each training/inference batch
BATCH_SIZE = 32

# Total number of target classes for the classification task
NUM_CLASSES = 10

In [3]:
# Define a function to create training and validation data loaders
def get_data_loaders(path, batch_size):
    """
    Creates PyTorch DataLoader objects for training and validation datasets.

    This function loads an image dataset from the specified directory using
    torchvision's ImageFolder, applies standard ImageNet preprocessing
    (resizing, tensor conversion, and normalization), and splits the dataset
    into training and validation subsets using an 80/20 ratio.

    Args:
        path (str): Path to the root directory of the image dataset. The directory
            must follow the ImageFolder structure, where each subdirectory
            represents a class label.
        batch_size (int): Number of samples per batch to be loaded.

    Returns:
        tuple:
            train_loader (torch.utils.data.DataLoader): DataLoader for the
                training dataset with shuffling enabled.
            val_loader (torch.utils.data.DataLoader): DataLoader for the
                validation dataset with shuffling disabled.
    """

    # Define a sequence of image transformations
    transform = transforms.Compose([
        # Resize all images to 224x224 to match ResNet input size
        transforms.Resize((224, 224)),
        # Convert PIL images to PyTorch tensors
        transforms.ToTensor(),
        # Normalize images using ImageNet mean and standard deviation
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    
    # Load the dataset from the directory structure using ImageFolder
    full_dataset = datasets.ImageFolder(root=path, transform=transform)
    
    # Compute the number of samples for the training set (80%)
    train_size = int(0.8 * len(full_dataset))
    
    # Compute the number of samples for the validation set (20%)
    val_size = len(full_dataset) - train_size
    
    # Randomly split the dataset into training and validation subsets
    train_ds, val_ds = torch.utils.data.random_split(
        full_dataset, [train_size, val_size]
    )
    
    # Create the training DataLoader with shuffling enabled
    # num_workers=2 is chosen for better stability on Kaggle
    train_loader = DataLoader(
        train_ds, batch_size=batch_size, shuffle=True, num_workers=2
    )
    
    # Create the validation DataLoader without shuffling
    val_loader = DataLoader(
        val_ds, batch_size=batch_size, shuffle=False, num_workers=2
    )
    
    # Return both training and validation data loaders
    return train_loader, val_loader

In [25]:
# ---------------------------------------------------------
# 2. Training Logic with Logging
# ---------------------------------------------------------

# Define a function to train a model and evaluate it after each epoch
def train_and_evaluate(model, train_loader, val_loader, optimizer, criterion, epochs, stage_name):
    """
    Trains a neural network model and evaluates its performance on a validation set.

    This function performs supervised training for a specified number of epochs.
    During each epoch, the model is trained using the provided training DataLoader,
    and after the epoch completes, evaluation metrics are computed on the validation
    dataset. Training progress, including loss and validation accuracy, is logged
    to the console.

    Args:
        model (torch.nn.Module): The neural network model to be trained.
        train_loader (torch.utils.data.DataLoader): DataLoader for the training dataset.
        val_loader (torch.utils.data.DataLoader): DataLoader for the validation dataset.
        optimizer (torch.optim.Optimizer): Optimizer used to update model parameters.
        criterion (torch.nn.Module): Loss function used to compute training loss.
        epochs (int): Number of training epochs to run.
        stage_name (str): Descriptive name of the training stage (e.g., "Pretraining",
            "Fine-tuning") used for logging purposes.

    Returns:
        dict: A dictionary containing evaluation metrics (e.g., accuracy) computed
        on the validation dataset after the final epoch.
    """

    # Print a header indicating the start of the training stage
    print(f"\n>>> Starting {stage_name} for {epochs} epochs...")
    
    # Loop over the specified number of training epochs
    for epoch in range(epochs):
        
        # Set the model to training mode (enables dropout and batch normalization)
        model.train()
        
        # Initialize a variable to accumulate training loss for the current epoch
        running_loss = 0.0
        
        # Iterate over batches of images and labels from the training DataLoader
        for images, labels in train_loader:
            
            # Move input images and target labels to the selected device (CPU/GPU)
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            # Reset gradients from the previous optimization step
            optimizer.zero_grad()
            
            # Perform a forward pass through the model
            outputs = model(images)
            
            # Compute the loss between predictions and ground-truth labels
            loss = criterion(outputs, labels)
            
            # Perform backpropagation to compute gradients
            loss.backward()
            
            # Update model parameters using the optimizer
            optimizer.step()
            
            # Accumulate the batch loss for epoch-level logging
            running_loss += loss.item()
            
        # Evaluate the model on the validation dataset after completing the epoch
        metrics = evaluate_metrics(model, val_loader)
        
        # Print epoch progress including average training loss and validation accuracy
        print(
            f"Epoch [{epoch+1}/{epochs}] - "
            f"Loss: {running_loss / len(train_loader):.4f} - "
            f"Val Acc: {metrics['Accuracy']:.4f}"
        )
    
    # Return the final evaluation metrics after training completes
    return metrics

In [26]:
# Define a function to compute evaluation metrics for a trained model
def evaluate_metrics(model, loader):
    """
    Computes classification performance metrics on a given dataset.

    This function switches the model to evaluation mode, performs inference
    without gradient computation, collects predictions and ground-truth labels,
    and calculates standard multi-class classification metrics including
    accuracy, precision, recall, and F1-score using macro averaging.

    Args:
        model (torch.nn.Module): Trained model used for inference.
        loader (torch.utils.data.DataLoader): DataLoader providing the evaluation dataset.

    Returns:
        dict: A dictionary containing the following evaluation metrics:
            - "Accuracy" (float): Overall classification accuracy.
            - "Precision" (float): Macro-averaged precision score.
            - "Recall" (float): Macro-averaged recall score.
            - "F1" (float): Macro-averaged F1-score.
    """

    # Set the model to evaluation mode (disables dropout and batch normalization updates)
    model.eval()
    
    # Initialize lists to store predictions and true labels
    all_preds, all_labels = [], []
    
    # Disable gradient computation to reduce memory usage and speed up inference
    with torch.no_grad():
        
        # Iterate over batches of images and labels from the DataLoader
        for images, labels in loader:
            
            # Move input images and labels to the selected device (CPU or GPU)
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            
            # Perform a forward pass through the model
            outputs = model(images)
            
            # Select the class with the highest predicted score for each sample
            _, preds = torch.max(outputs, dim=1)
            
            # Move predictions to CPU and convert them to NumPy arrays
            all_preds.extend(preds.cpu().numpy())
            
            # Move ground-truth labels to CPU and convert them to NumPy arrays
            all_labels.extend(labels.cpu().numpy())
            
    # Compute overall classification accuracy
    acc = accuracy_score(all_labels, all_preds)
    
    # Compute macro-averaged precision, recall, and F1-score
    # zero_division=0 avoids errors when a class has no predicted samples
    p, r, f1, _ = precision_recall_fscore_support(
        all_labels, all_preds, average='macro', zero_division=0
    )
    
    # Return evaluation metrics as a dictionary
    return {
        "Accuracy": acc,
        "Precision": p,
        "Recall": r,
        "F1": f1
    }

In [11]:
# ---------------------------------------------------------
# 3. Execution Logic
# ---------------------------------------------------------

# Initialize a ResNet-18 model with ImageNet pre-trained weights
model = models.resnet18(weights='IMAGENET1K_V1')

# Replace the final fully connected layer to match the number of target classes
model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)

# Move the model to the selected computation device (CPU or GPU)
model = model.to(DEVICE)

# Create training and validation data loaders from the dataset
train_loader, val_loader = get_data_loaders(DATA_PATH, BATCH_SIZE)

# Define the loss function for multi-class classification
criterion = nn.CrossEntropyLoss()

# Initialize a list to store final evaluation results
final_results = []

In [28]:
# --- STAGE 1: Head Only (5 Epochs) ---

# Freeze all model parameters to prevent them from being updated during training
for param in model.parameters():
    param.requires_grad = False

# Unfreeze only the parameters of the final fully connected layer (classification head)
for param in model.fc.parameters():
    param.requires_grad = True

# Initialize the Adam optimizer to update only the classification head parameters
optimizer = optim.Adam(model.fc.parameters(), lr=1e-3)

# Train and evaluate the model for 5 epochs using only the classification head
res1 = train_and_evaluate(
    model, train_loader, val_loader, optimizer, criterion, 5, "Stage 1: Head Only"
)

# Add a label to identify the training stage in the results dictionary
res1["Stage"] = "1. Head Only"

# Store the results of Stage 1 for later comparison or reporting
final_results.append(res1)



>>> Starting Stage 1: Head Only for 5 epochs...
Epoch [1/5] - Loss: 1.7471 - Val Acc: 0.6860
Epoch [2/5] - Loss: 1.0512 - Val Acc: 0.7180
Epoch [3/5] - Loss: 0.8282 - Val Acc: 0.7560
Epoch [4/5] - Loss: 0.7347 - Val Acc: 0.7520
Epoch [5/5] - Loss: 0.6653 - Val Acc: 0.7640


In [29]:
# --- STAGE 2: Layer 4 (3 Epochs) ---

# Unfreeze the parameters of layer4 to allow fine-tuning of high-level features
for param in model.layer4.parameters():
    param.requires_grad = True

# Initialize the Adam optimizer for all trainable parameters (layer4 + classification head)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4
)

# Train and evaluate the model for 3 epochs with layer4 and head unfrozen
res2 = train_and_evaluate(
    model, train_loader, val_loader, optimizer, criterion, 3, "Stage 2: Layer 4 + Head"
)

# Add a label to identify the training stage in the results dictionary
res2["Stage"] = "2. Layer 4 + Head"

# Store the results of Stage 2 for later comparison or reporting
final_results.append(res2)


>>> Starting Stage 2: Layer 4 + Head for 3 epochs...
Epoch [1/3] - Loss: 0.5222 - Val Acc: 0.8400
Epoch [2/3] - Loss: 0.1055 - Val Acc: 0.8600
Epoch [3/3] - Loss: 0.0407 - Val Acc: 0.8420


In [30]:
# --- STAGE 3: Full Fine-Tune (3 Epochs) ---

# Unfreeze all model parameters to enable full fine-tuning
for param in model.parameters():
    param.requires_grad = True

# Initialize the Adam optimizer for all model parameters with a low learning rate
optimizer = optim.Adam(model.parameters(), lr=1e-5)

# Train and evaluate the entire model for 3 epochs using full fine-tuning
res3 = train_and_evaluate(
    model, train_loader, val_loader, optimizer, criterion, 3, "Stage 3: Full Fine-Tune"
)

# Add a label to identify the training stage in the results dictionary
res3["Stage"] = "3. Full Fine-Tune"

# Store the results of Stage 3 for later comparison or reporting
final_results.append(res3)


>>> Starting Stage 3: Full Fine-Tune for 3 epochs...
Epoch [1/3] - Loss: 0.0230 - Val Acc: 0.8500
Epoch [2/3] - Loss: 0.0154 - Val Acc: 0.8520
Epoch [3/3] - Loss: 0.0114 - Val Acc: 0.8560


In [31]:
# ---------------------------------------------------------
# 4. Final Summary
# ---------------------------------------------------------

# Convert the list of final evaluation results into a Pandas DataFrame
df = pd.DataFrame(final_results)

# Print a visual separator before the final report
print("\n" + "=" * 30)

# Print the title of the final transfer learning report
print("FINAL TRANSFER LEARNING REPORT")

# Print a visual separator after the report title
print("=" * 30)

# Display selected evaluation metrics for each training stage
print(df[["Stage", "Accuracy", "Precision", "Recall", "F1"]])


FINAL TRANSFER LEARNING REPORT
               Stage  Accuracy  Precision    Recall        F1
0       1. Head Only     0.764   0.773284  0.764940  0.761035
1  2. Layer 4 + Head     0.842   0.843922  0.843715  0.842464
2  3. Full Fine-Tune     0.856   0.854929  0.856444  0.855357


---
**Note:** This notebook is part of a Deep Learning assignment designed and prepared by [Mahdi Golizadeh](mailto:mahdi.golizadeh@gmail.com).